In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/sinayavarian
/kaggle/input/datasets/sinayavarian/person-detection-yolo11
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/valid
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/valid/labels
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/valid/images
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/test
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/labels
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/images
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/train
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/train/labels
/kaggle/input/datasets/sinayavarian/person-detection-yolo11/train/images


In [4]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU")

True
Tesla T4


In [5]:
!pip -q install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.1 MB/s eta 0:00:00a 0:00:01


In [10]:
from pathlib import Path

DATASET = Path(
    "/kaggle/input/datasets/sinayavarian/person-detection-yolo11"
)

for split in ["train", "valid", "test"]:
    images = list((DATASET / split / "images").glob("*"))
    labels = list((DATASET / split / "labels").glob("*.txt"))

    print(
        split,
        "images:", len(images),
        "labels:", len(labels)
    )

train images: 12231 labels: 12231
valid images: 899 labels: 899
test images: 846 labels: 846


In [11]:
from pathlib import Path

for p in DATASET.rglob("*.yaml"):
    print(p)

/kaggle/input/datasets/sinayavarian/person-detection-yolo11/data.yaml


In [12]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data="/kaggle/input/datasets/sinayavarian/person-detection-yolo11/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="/kaggle/working",
    name="person_detection_yolo11"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/sinayavarian/person-detection-yolo11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, flip

Exception in thread Thread-12 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

KeyboardInterrupt: 

In [13]:
from pathlib import Path

weights = Path("/kaggle/working/person_detection_yolo11/weights")

print(weights.exists())

if weights.exists():
    for f in weights.iterdir():
        print(f.name)

True
best.pt
last.pt


In [14]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/person_detection_yolo11/weights/best.pt")

metrics = model.val()

print(metrics)

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 181.7±39.6 MB/s, size: 101.5 KB)
val: Scanning /kaggle/input/datasets/sinayavarian/person-detection-yolo11/valid/labels... 899 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 899/899 654.2it/s 1.4s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/sinayavarian/person-detection-yolo11/valid is not writable, cache not saved.
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 26, len(boxes) = 8948. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 5.1it/s 11.3s0.2s
                   all        899       

In [15]:
results = model.predict(
    source="/kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/images",
    save=True,
    conf=0.25
)


image 1/846 /kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/images/20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed5044d861eacf.jpg: 640x640 5 Heads, 5 Persons, 9.2ms
image 2/846 /kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/images/20221013_141932_jpg.rf.2dc49953531da1a64d29602ecf063da4.jpg: 640x640 8 Heads, 6 Persons, 9.7ms
image 3/846 /kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/images/20230313_102836_jpg.rf.15e8d2ede3dc03f425d29602507028fb.jpg: 640x640 3 Heads, 3 Persons, 9.6ms
image 4/846 /kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/images/20230314_151719_jpg.rf.2abca3dec8ed6609e630ee7848de547e.jpg: 640x640 6 Heads, 7 Persons, 10.0ms
image 5/846 /kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/images/2024-03-25-at-17-16-24_f994c8cb-frame-018_jpg.rf.14a3e2135bfcd668002588def85ff200.jpg: 640x640 2 Heads, 2 Persons, 9.4ms
image 6/846 /kaggle/input/datasets/sinayavarian/person-detection-yolo11/test/

In [16]:
import pandas as pd
from pathlib import Path

rows = []

for result in results:
    image_name = Path(result.path).name
    boxes = result.boxes

    row = {
        "image": image_name,
        "total_detections": 0,
        "head_count": 0,
        "person_count": 0,
    }

    if boxes is not None and len(boxes) > 0:
        class_ids = boxes.cls.cpu().numpy().astype(int)
        class_names = result.names

        row["total_detections"] = len(class_ids)
        row["head_count"] = sum(
            class_names[class_id].lower() == "head"
            for class_id in class_ids
        )
        row["person_count"] = sum(
            class_names[class_id].lower() == "person"
            for class_id in class_ids
        )

    rows.append(row)

detection_table = pd.DataFrame(rows)

display(detection_table.head(20))

,image,total_detections,head_count,person_count
0,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,10,5,5
1,20221013_141932_jpg.rf.2dc49953531da1a64d29602...,14,8,6
2,20230313_102836_jpg.rf.15e8d2ede3dc03f425d2960...,6,3,3
3,20230314_151719_jpg.rf.2abca3dec8ed6609e630ee7...,13,6,7
4,2024-03-25-at-17-16-24_f994c8cb-frame-018_jpg....,4,2,2
5,2024-03-25-at-17-16-24_f994c8cb-frame-030_jpg....,4,2,2
6,2024-03-25-at-17-16-33_89f300fc-frame-013_jpg....,9,4,5
7,2024-03-25-at-17-16-34_367f09d6-frame-016_jpg....,2,1,1
8,20240320_093023_jpg.rf.e63e516ce8a074084a9bd43...,4,2,2
9,20240321_092805_jpg.rf.0ad2fd20e920e7cd09187a6...,24,11,13


In [17]:
summary_table = pd.DataFrame({
    "Metric": [
        "Number of test images",
        "Total detections",
        "Total Head detections",
        "Total Person detections",
        "Images with detections",
        "Images without detections",
        "Average detections per image",
    ],
    "Value": [
        len(detection_table),
        detection_table["total_detections"].sum(),
        detection_table["head_count"].sum(),
        detection_table["person_count"].sum(),
        (detection_table["total_detections"] > 0).sum(),
        (detection_table["total_detections"] == 0).sum(),
        detection_table["total_detections"].mean(),
    ],
})

display(summary_table)

,Metric,Value
0,Number of test images,846.000000
1,Total detections,8535.000000
2,Total Head detections,3850.000000
3,Total Person detections,4685.000000
4,Images with detections,827.000000
5,Images without detections,19.000000
6,Average detections per image,10.088652


In [18]:
confidence_rows = []

for result in results:
    image_name = Path(result.path).name
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        continue

    class_ids = boxes.cls.cpu().numpy().astype(int)
    confidences = boxes.conf.cpu().numpy()
    coordinates = boxes.xyxy.cpu().numpy()

    for class_id, confidence, box in zip(
        class_ids,
        confidences,
        coordinates,
    ):
        confidence_rows.append({
            "image": image_name,
            "class": result.names[class_id],
            "confidence": float(confidence),
            "x1": float(box[0]),
            "y1": float(box[1]),
            "x2": float(box[2]),
            "y2": float(box[3]),
        })

confidence_table = pd.DataFrame(confidence_rows)

display(confidence_table.head(20))

,image,class,confidence,x1,y1,x2,y2
0,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Person,0.829953,164.532715,194.520935,230.733398,533.714966
1,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Head,0.819055,318.326843,184.479797,343.188232,232.934509
2,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Person,0.805144,297.591492,184.804947,360.835693,516.057617
3,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Head,0.800328,177.781967,195.862701,210.602341,247.527496
4,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Person,0.787434,367.397583,209.703491,420.227905,546.029907
5,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Head,0.782486,383.368439,208.561462,410.457733,266.267273
6,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Head,0.647308,345.330505,209.277405,363.162415,254.001129
7,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Head,0.572394,245.237793,159.070053,264.696991,194.490341
8,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Person,0.494610,235.380768,163.120789,270.399445,409.503479
9,20220525_075311_jpg.rf.2d4d7515ee7a1f1563ed504...,Person,0.446907,350.220886,209.652740,375.752014,479.698151


In [19]:
class_confidence_summary = (
    confidence_table
    .groupby("class")
    .agg(
        detections=("confidence", "count"),
        mean_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
        minimum_confidence=("confidence", "min"),
        maximum_confidence=("confidence", "max"),
    )
    .reset_index()
)

class_confidence_summary[
    [
        "mean_confidence",
        "median_confidence",
        "minimum_confidence",
        "maximum_confidence",
    ]
] = class_confidence_summary[
    [
        "mean_confidence",
        "median_confidence",
        "minimum_confidence",
        "maximum_confidence",
    ]
].round(3)

display(class_confidence_summary)

,class,detections,mean_confidence,median_confidence,minimum_confidence,maximum_confidence
0,Head,3850,0.720,0.782,0.251,0.914
1,Person,4685,0.697,0.786,0.250,0.933


In [20]:
speed_rows = []

for result in results:
    speed_rows.append({
        "image": Path(result.path).name,
        "preprocess_ms": result.speed.get("preprocess", 0),
        "inference_ms": result.speed.get("inference", 0),
        "postprocess_ms": result.speed.get("postprocess", 0),
    })

speed_table = pd.DataFrame(speed_rows)

speed_summary = pd.DataFrame({
    "Stage": [
        "Preprocessing",
        "Inference",
        "Postprocessing",
        "Total",
    ],
    "Average time per image (ms)": [
        speed_table["preprocess_ms"].mean(),
        speed_table["inference_ms"].mean(),
        speed_table["postprocess_ms"].mean(),
        (
            speed_table["preprocess_ms"]
            + speed_table["inference_ms"]
            + speed_table["postprocess_ms"]
        ).mean(),
    ],
})

speed_summary["Average time per image (ms)"] = (
    speed_summary["Average time per image (ms)"].round(2)
)

display(speed_summary)

,Stage,Average time per image (ms)
0,Preprocessing,2.58
1,Inference,9.41
2,Postprocessing,1.56
3,Total,13.55


In [21]:
average_total_ms = (
    speed_table["preprocess_ms"]
    + speed_table["inference_ms"]
    + speed_table["postprocess_ms"]
).mean()

estimated_fps = 1000 / average_total_ms

fps_table = pd.DataFrame({
    "Metric": ["Average processing time", "Estimated FPS"],
    "Value": [
        f"{average_total_ms:.2f} ms/image",
        f"{estimated_fps:.2f} FPS",
    ],
})

display(fps_table)

,Metric,Value
0,Average processing time,13.55 ms/image
1,Estimated FPS,73.80 FPS


In [22]:
OUTPUT_DIR = Path("/kaggle/working/yolo_report_tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

detection_table.to_csv(
    OUTPUT_DIR / "test_image_detection_counts.csv",
    index=False,
)

summary_table.to_csv(
    OUTPUT_DIR / "test_detection_summary.csv",
    index=False,
)

confidence_table.to_csv(
    OUTPUT_DIR / "all_test_predictions.csv",
    index=False,
)

class_confidence_summary.to_csv(
    OUTPUT_DIR / "class_confidence_summary.csv",
    index=False,
)

speed_table.to_csv(
    OUTPUT_DIR / "inference_speed_per_image.csv",
    index=False,
)

speed_summary.to_csv(
    OUTPUT_DIR / "inference_speed_summary.csv",
    index=False,
)

fps_table.to_csv(
    OUTPUT_DIR / "estimated_fps.csv",
    index=False,
)

print("Tables saved to:", OUTPUT_DIR)

Tables saved to: /kaggle/working/yolo_report_tables
